In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # pick a GPU before importing chrombpnet/jax

import chrombpnet  # selects the Keras JAX backend; import it before keras
import chrombpnet.training.utils.one_hot as one_hot
from chrombpnet.training.utils.model_io import load_model
from chrombpnet.evaluation.interpret.explainer import DeepLiftShap
import matplotlib.pyplot as plt
import viz_sequence

In [ ]:
# ---- Step 1: Load model ----

MODEL_PATH = "results/chrombpnet/ATAC_PE/GM12878/nautilus_runs/GM12878_03.01.2022_bias_128_4_1234_0.4_fold_0/chrombpnet_model/chrombpnet_wo_bias.h5"  # Change this path to your actual model
model = load_model(MODEL_PATH)  # chrombpnet 1.x (TF-Keras) and 2.x .h5 files

In [ ]:
# ---- Step 2: Input your custom 2114bp sequence ----

# Make sure it only contains A, C, G, T
def load_sequences_from_fasta(fasta_path, seq_len):
    sequences = []
    with open(fasta_path) as f:
        current_seq = []
        for line in f:
            if line.startswith(">"):
                if current_seq:
                    sequences.append("".join(current_seq))
                    current_seq = []
            else:
                current_seq.append(line.strip())

        if current_seq:
            sequences.append("".join(current_seq))
    assert all(len(s) == seq_len for s in sequences), "Input sequences must be exactly {} bp long.".format(seq_len)
    return sequences

sequences = load_sequences_from_fasta("example.fa", model.input_shape[1])

In [ ]:
# ---- Step 3: One-hot encode the sequence ----

one_hot_seqs = one_hot.dna_to_one_hot(sequences)  # shape: (N, 2114, 4)
print("One-hot shape:", one_hot_seqs.shape)


In [ ]:
# Get count shap scores
# 20 dinucleotide-shuffled references per sequence, seeded from `seed` and the sequence itself (reproducible)
counts_explainer = DeepLiftShap(model, heads=["counts"])

#  counts_shap_scores has final hypothetical contribution scores from counts head
print("Generating 'counts' shap scores")
counts_shap_scores = counts_explainer.explain(one_hot_seqs, seed=1234)["counts"]

In [ ]:
counts_shap_scores.shape

In [ ]:

index=0 # plot contributions for example 0

#  Multiple counts_shap_scores with one_hot_seqs to get observed shap scores on counts head

observed_counts_scores = counts_shap_scores * one_hot_seqs

ymin = counts_shap_scores[index].min()
ymax = counts_shap_scores[index].max()
viz_sequence.plot_weights(observed_counts_scores[index, 1057-100:1057+100,:],subticks_frequency=5,ylim=[ymin,ymax])
plt.show() 
    

In [ ]:
# Get profile shap scores

profile_explainer = DeepLiftShap(model, heads=["profile"])

#  profile_shap_scores has final hypothetical contribution scores from profile head
print("Generating 'profile' shap scores")
profile_shap_scores = profile_explainer.explain(one_hot_seqs, seed=1234)["profile"]

In [ ]:
index=0 # plot contributions for example 0

#  Multiple profile_shap_scores with one_hot_seqs to get observed shap scores on profile head

observed_profile_scores = profile_shap_scores * one_hot_seqs

ymin = observed_profile_scores[index].min()
ymax = observed_profile_scores[index].max()
viz_sequence.plot_weights(observed_profile_scores[index, 1057-100:1057+100,:],subticks_frequency=5,ylim=[ymin,ymax])
plt.show() 
    